## Configuration


In [0]:

class Config():
    def __init__(self):
        self.db_name = "workout"
        self.max_FilesPerTrigger = 100

In [0]:
config = Config()
print(f"Database Name: {config.db_name}")

## Set Up


In [1]:
import os
def get_spark():
    """Get Spark session based on environment"""
    
    if "DATABRICKS_RUNTIME_VERSION" in os.environ:
        print("Running in Databricks environment")
        from databricks.sdk.runtime import spark
        return spark
    else:
        print("Running locally with Databricks Connect")
        from databricks.connect import DatabricksSession
        spark = (DatabricksSession
                .builder
                .profile("dev-free-edition")
                .serverless(True)
                .getOrCreate())
        return spark


spark = get_spark()
spark

Running locally with Databricks Connect


In [0]:

class SetUp():
    def __init__(self,env):
        conf = Config()
        self.db_name = conf.db_name
        self.catalog = env
        self.initialized = False


    def create_db(self):
        print(f"Creating the database {self.catalog}.{self.db_name}...", end='')
        spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.catalog}.{self.db_name}")
        spark.sql(f"USE {self.catalog}.{self.db_name}")
        self.initialized = True
        print("Done.")













In [0]:
setup = SetUp(env="dev")


In [0]:
setup.create_db()

In [0]:
%sql

show databases in dev;

In [0]:

def get_secrets(name: str,  scope: str = "test", env: str = "local"):
    """Get secrets from environment variables or Databricks Secrets"""

    if env == "local":

        from dotenv import load_dotenv
        import os

        load_dotenv()
        return os.getenv(name)
    
    try:
        return dbutils.secrets.get(scope=scope, key=name)
    except Exception as e:
        print(f"Error getting secret {name} from scope {scope}: {e}")

    raise ValueError(f"Secret {name} not found in scope {scope}")


In [0]:
azure_storage_account = get_secrets("azure_storage_account")


In [3]:
catalog = "dev"

ingestion_db = "control"
ingestion_table = "ingest_table_registry"
bronze_db = "bronze_workout"

schema_registry_table = "schema_registry"

schema_registry_table = "schema_registry"
history_table = "ingest_run_history"
# spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{ingestion_db};")

# spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{bronze_db};")

In [4]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StructType
from pyspark.sql import SparkSession, DataFrame
def get_defined_schema(table_id: int,spark: SparkSession):
    schema = (
        spark.table(f"{catalog}.{ingestion_db}.{schema_registry_table}")
        .filter(f"table_id = {table_id} AND is_latest = TRUE")
        .orderBy(F.desc("version"))
        .limit(1)
        .collect()[0]
    )

    return StructType.fromJson(json.loads(schema.schema_definition))



In [5]:
schema = get_defined_schema(2,spark)
schema

StructType([StructField('mac_address', StringType(), True), StructField('gym', StringType(), True), StructField('login', StringType(), True), StructField('logout', StringType(), True)])

In [6]:
def load_date(spark, configs_dict, schema) -> DataFrame:
    """Load data from a given path with the provided schema."""

    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", configs_dict["source_format"])
        .option("cloudFiles.schemaLocation", configs_dict["checkpoint_path"] + "schema/" + configs_dict["target_table"])
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("cloudFiles.maxFilesPerTrigger", 1)
        .schema(schema)
        .load(configs_dict["source_path"])
        .withColumn("ingestion_timestamp", F.current_timestamp())

    )

In [11]:
df = load_date(spark, configs_dict, schema)


In [12]:
view_streaming_df(df)

AnalysisException: Queries with streaming sources must be executed with writeStream.start(), or from a streaming table or flow definition within a Lakeflow Declarative Pipeline.;
cloudFiles

JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.catalyst.analysis.UnsupportedOperationChecker$.throwError(UnsupportedOperationChecker.scala:719)
	at org.apache.spark.sql.catalyst.analysis.UnsupportedOperationChecker$.$anonfun$checkForBatch$2(UnsupportedOperationChecker.scala:68)
	at org.apache.spark.sql.catalyst.analysis.UnsupportedOperationChecker$.$anonfun$checkForBatch$2$adapted(UnsupportedOperationChecker.scala:65)
	at org.apache.spark.sql.catalyst.trees.TreeNode.foreachUp(TreeNode.scala:325)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$foreachUp$1(TreeNode.scala:324)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$foreachUp$1$adapted(TreeNode.scala:324)
	at scala.collection.immutable.Vector.foreach(Vector.scala:2125)
	at o

DataFrame[mac_address: string, gym: string, login: string, logout: string, _rescued_data: string, ingestion_timestamp: timestamp]

In [13]:
dbutils.fs.rm("/Volumes/dev/bronze_workout/checkpoints/")

NameError: name 'dbutils' is not defined

In [9]:
configs_dict = [d for d in configs_dict if d["table_id"] == 2][0]
configs_dict

{'table_id': 2,
 'source_path': '/Volumes/dev/bronze_workout/rw/dataset/gym_logins/',
 'source_format': 'csv',
 'header': True,
 'target_catalog': 'dev',
 'target_schema': 'bronze_workout',
 'table_version': 1,
 'target_table': 'gym_logins',
 'write_mode': 'append',
 'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/',
 'merge_key': [],
 'partition_by': [],
 'enabled': True,
 'trigger_mode': 'availableNow',
 'processing_time': '5 minutes',
 'schema_evolution_mode': 'BACKWARD'}

In [15]:
table_id_automatic = spark.sql(f"select distinct table_id from {catalog}.{ingestion_db}.{schema_registry_table}")
table_id_automatic

,table_id
0,1


In [17]:
[i.table_id for i in table_id_automatic.collect()]

[1]

In [20]:
spark.sql(f"select distinct table_id as table_id from {catalog}.{ingestion_db}.{ingestion_table} where enabled = true").show()

+--------+
|table_id|
+--------+
|       1|
|       2|
+--------+



In [7]:
def get_configs(catalog, ingestion_db, ingestion_table, spark):
    query = f"""
    SELECT *
    FROM {catalog}.{ingestion_db}.{ingestion_table}
    WHERE enabled = TRUE
    """
    df = spark.sql(query)
    if df.isEmpty():
        raise ValueError(f"No configurations found for table_ids")

    configs_dict = (
        df.select(
            "table_id",
            "source_path",
            "source_format",
            "header",
            "target_catalog",
            "target_schema",
            "table_version",
            "target_table",
            "write_mode",
            "checkpoint_path",
            "merge_key",
            "partition_by",
            "enabled",
            "trigger_mode",
            "processing_time",
            "schema_evolution_mode",
        )
    )

    return [row.asDict() for row in configs_dict.collect()]


In [8]:
configs_dict = get_configs(catalog, ingestion_db, ingestion_table, spark)
configs_dict

[{'table_id': 2,
  'source_path': '/Volumes/dev/bronze_workout/rw/dataset/gym_logins/',
  'source_format': 'csv',
  'header': True,
  'target_catalog': 'dev',
  'target_schema': 'bronze_workout',
  'table_version': 1,
  'target_table': 'gym_logins',
  'write_mode': 'append',
  'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/',
  'merge_key': [],
  'partition_by': [],
  'enabled': True,
  'trigger_mode': 'availableNow',
  'processing_time': '5 minutes',
  'schema_evolution_mode': 'BACKWARD'},
 {'table_id': 1,
  'source_path': '/Volumes/dev/bronze_workout/rw/dataset/bpm/',
  'source_format': 'json',
  'header': True,
  'target_catalog': 'dev',
  'target_schema': 'bronze_workout',
  'table_version': 1,
  'target_table': 'bpm',
  'write_mode': 'append',
  'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/',
  'merge_key': [],
  'partition_by': [],
  'enabled': True,
  'trigger_mode': 'availableNow',
  'processing_time': '5 minutes',
  'schema_evolution_mode': 'BACKWAR

In [19]:
table_ids = [conf["table_id"] for conf in configs_dict]
table_ids

[1, 2]

In [5]:
[d for d in configs_dict if d["table_id"] == 2][0]


{'table_id': 2,
 'source_path': '/Volumes/dev/bronze_workout/rw/dataset/gym_logins/',
 'source_format': 'json',
 'header': True,
 'target_catalog': 'dev',
 'target_schema': 'bronze_workout',
 'table_version': 1,
 'target_table': 'gym_logins',
 'write_mode': 'append',
 'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/',
 'merge_key': [],
 'partition_by': [],
 'enabled': True,
 'trigger_mode': 'availableNow',
 'processing_time': '5 minutes',
 'schema_evolution_mode': 'BACKWARD'}

In [5]:
spark.table("dev.control.ingest_table_registry").show(truncate=False)

+--------+--------------------------------------------------+-------------+------+-------------+--------------+--------------+------------+----------+----------------------------------------+---------+------------+-------+------------+---------------+---------------------+---------------------+--------------------------+----------+--------------------+-------------------------+---------------+--------------------------+----------+--------------------------+
|table_id|source_path                                       |source_format|header|table_version|target_catalog|target_schema |target_table|write_mode|checkpoint_path                         |merge_key|partition_by|enabled|trigger_mode|processing_time|schema_evolution_mode|expected_rows_per_day|last_success              |last_error|consecutive_failures|last_run_duration_seconds|created_by     |created_at                |updated_by|updated_at                |
+--------+--------------------------------------------------+-------------+-

In [ ]:
spark.sql("UPDATE dev.control.ingest_table_registry set source_format = 'csv'")

In [2]:
spark.read.format("json").load("/Volumes/dev/bronze_workout/rw/dataset/bpm/").show()

+------+------+---------+----------+-----+--------------------+
|   key|offset|partition| timestamp|topic|               value|
+------+------+---------+----------+-----+--------------------+
|118440|     0|        0|1678410000|  bpm|{118440, 41.21897...|
|118440|     1|        1|1678410001|  bpm|{118440, 46.95949...|
|118440|     2|        2|1678410002|  bpm|{118440, 84.39302...|
|118440|     3|        3|1678410003|  bpm|{118440, 87.49022...|
|118440|     4|        4|1678410004|  bpm|{118440, 76.16501...|
|118440|     5|        5|1678410005|  bpm|{118440, 42.46430...|
|118440|     6|        6|1678410006|  bpm|{118440, 46.98973...|
|118440|     7|        7|1678410007|  bpm|{118440, 93.72495...|
|118440|     8|        8|1678410008|  bpm|{118440, 85.67159...|
|118440|     9|        9|1678410009|  bpm|{118440, 80.44560...|
|118440|    10|       10|1678410010|  bpm|{118440, 56.13216...|
|118440|    11|       11|1678410011|  bpm|{118440, 45.26557...|
|118440|    12|       12|1678410012|  bp

In [9]:
header = 'true'

In [10]:
spark.read.format("csv").load("/Volumes/dev/bronze_workout/rw/dataset/gym_logins/",header=header.title()).show()

+-----------------+---+----------+----------+
|      mac_address|gym|     login|    logout|
+-----------------+---+----------+----------+
|4c:c5:9f:cb:13:bd|  5|1678521600|1678526100|
|ae:ec:f6:48:ca:f7|  1|1678522500|1678525200|
|36:1f:d9:d3:e8:0d|  3|1678522500|1678527000|
|14:cd:d6:db:70:f6|  5|1678523400|1678527600|
|57:24:ac:8c:75:ea|  1|1678524000|1678528500|
|36:1f:d9:d3:e8:0d|  3|1678561200|1678564800|
|14:cd:d6:db:70:f6|  5|1678562400|1678565700|
|57:24:ac:8c:75:ea|  5|1678562880|1678567200|
|1d:69:69:75:d0:aa|  1|1678608000|1678611000|
|df:f9:dc:5e:e2:a8|  1|1678608000|1678611600|
|dd:96:be:e9:1e:f4|  3|1678608720|1678611600|
|dd:45:d2:37:a8:0e|  5|1678609020|1678612320|
|de:c0:cd:a7:71:f4|  5|1678609800|1678614000|
|1d:69:69:75:d0:aa|  3|1678644000|1678646400|
|df:f9:dc:5e:e2:a8|  3|1678644720|1678647300|
|dd:96:be:e9:1e:f4|  1|1678645440|1678649700|
+-----------------+---+----------+----------+



In [20]:
table_ids = dbutils.jobs.taskValues.get(
    taskKey="Ingestion_master",   # task_key of master
    key="table_ids",
    debugValue=[]
)

print(table_ids)


[]


In [4]:
insert_query = f"""

INSERT INTO {catalog}.{ingestion_db}.{ingestion_table} 
(source_path,                         source_format,header,table_version,target_catalog,target_schema,target_table,write_mode,checkpoint_path,merge_key,partition_by,enabled,trigger_mode,processing_time,schema_evolution_mode,expected_rows_per_day,last_success,last_error,consecutive_failures,last_run_duration_seconds,created_by,created_at,updated_by,updated_at)
VALUES 
('/Volumes/dev/bronze_workout/rw/dataset/gym_logins/', 'json', TRUE, 1, 'dev', 'bronze_workout', 'gym_logins', 'append', '/Volumes/dev/bronze_workout/checkpoints/', ARRAY(), ARRAY(), TRUE, 'availableNow', '5 minutes', 'BACKWARD', 1000000, NULL, NULL, 0, NULL, 'gaurav.thagunna', CURRENT_TIMESTAMP(), NULL, CURRENT_TIMESTAMP());

"""

spark.sql(insert_query)

,num_affected_rows,num_inserted_rows
0,1,1


In [0]:
query = f"""

CREATE TABLE IF NOT EXISTS {catalog}.{ingestion_db}.{ingestion_table} (
  table_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  source_path STRING NOT NULL,
  source_format STRING DEFAULT 'parquet',
  header BOOLEAN DEFAULT TRUE,
  table_version INT DEFAULT 1,

  target_catalog STRING DEFAULT 'dev',
  target_schema STRING DEFAULT 'test',
  target_table STRING NOT NULL,
  write_mode STRING DEFAULT 'append'
      CHECK (write_mode IN ('append', 'merge', 'overwrite')),
  checkpoint_path STRING NOT NULL,
  merge_key ARRAY<STRING> DEFAULT ARRAY(),
  partition_by ARRAY<STRING> DEFAULT ARRAY(),
  enabled BOOLEAN DEFAULT TRUE,

 trigger_mode STRING DEFAULT 'availableNow'
    CHECK (trigger_mode IN ('availableNow', 'processingTime', 'continuous')),
  processing_time STRING DEFAULT '5 minutes',
  schema_evolution_mode STRING DEFAULT 'BACKWARD'
    CHECK (schema_evolution_mode IN ('BACKWARD', 'FORWARD', 'FULL', 'NONE')),
  
  expected_rows_per_day BIGINT,
  last_success TIMESTAMP,
  last_error STRING,
  consecutive_failures INT,
  last_run_duration_seconds INT,
  created_by STRING,
  created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  updated_by STRING,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.autoCompact' = 'true',
  'delta.feature.allowColumnDefaults' = 'supported'
);

"""

In [0]:
spark.sql(query)

In [0]:
insert_query = f"""

INSERT INTO {catalog}.{ingestion_db}.{ingestion_table} 
(source_path,                         source_format,header,table_version,target_catalog,target_schema,target_table,write_mode,checkpoint_path,merge_key,partition_by,enabled,trigger_mode,processing_time,schema_evolution_mode,expected_rows_per_day,last_success,last_error,consecutive_failures,last_run_duration_seconds,created_by,created_at,updated_by,updated_at)
VALUES 
('/Volumes/dev/bronze_workout/rw/dataset/bpm/', 'json', TRUE, 1, 'dev', 'bronze_workout', 'bpm', 'append', '/Volumes/dev/bronze_workout/checkpoints/', ARRAY(), ARRAY(), TRUE, 'availableNow', '5 minutes', 'BACKWARD', 1000000, NULL, NULL, 0, NULL, 'gaurav.thagunna', CURRENT_TIMESTAMP(), NULL, CURRENT_TIMESTAMP());

"""

spark.sql(insert_query)

In [0]:
configs = spark.sql(f"select * from {catalog}.{ingestion_db}.{ingestion_table}")
configs.show(truncate=False)

In [0]:
configs_dict = configs.select("table_id","source_path","source_format","header","target_catalog","target_schema","table_version","target_table","write_mode","checkpoint_path","merge_key","partition_by","enabled","trigger_mode","processing_time","schema_evolution_mode").collect()[0].asDict()
configs_dict

In [0]:
def get_schema_file(path:str):
    df = spark.read.json(path)
    return df.schema


In [0]:
schema = get_schema_file(path=configs_dict["source_path"])
schema

In [0]:
ingestion_db = "control"
schema_registry_tb = "schema_registry"

schema_registry_query = f"""CREATE TABLE IF NOT EXISTS {catalog}.{ingestion_db}.{schema_registry_tb} (
  id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  table_id INT NOT NULL,
  table_name STRING NOT NULL,
  version INT DEFAULT 1,
  is_latest BOOLEAN DEFAULT TRUE,                 
  schema_format STRING,          
  schema_definition STRING,     
  created_by STRING,
  created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP()
  )
  TBLPROPERTIES (
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.feature.allowColumnDefaults' = 'supported'
  );

  """

spark.sql(schema_registry_query)

In [0]:
spark.table(f"{catalog}.{ingestion_db}.{schema_registry_tb}").show()

In [0]:
spark.sql(f"SELECT * FROM {catalog}.{ingestion_db}.{schema_registry_tb}").show(truncate=False)

In [0]:
query = f"""

INSERT INTO {catalog}.{ingestion_db}.{schema_registry_tb} (
  table_id,
  table_name,
  version,
  is_latest,
  schema_format,
  schema_definition,
  created_by
)
VALUES (
  {configs_dict["table_id"]},
  '{configs_dict["target_table"]}',
  1,
  TRUE,
  'json',
  '{schema.json()}',
  'gaurav.thagunna'
)

"""

spark.sql(query)



NameError: name 'catalog' is not defined

In [0]:
spark.table("dev.control.schema_registry").show(truncate=False)

In [0]:
schema_row = (
    spark.table("dev.control.schema_registry")
         .filter("table_id = 1 AND is_latest = TRUE").limit(1).collect()[0]


)
schema_row

In [0]:
from pyspark.sql.types import StructType
import json
schema = StructType.fromJson(json.loads(schema_row.schema_definition))
schema

In [0]:
spark.read.json(configs_dict["source_path"]).show(truncate=False)

In [ ]:
        - task_key: Ingest_for_each_table
          depends_on:
            - task_key: Schema_Registry
          for_each_task:
            inputs: "{{tasks.Ingestion_Master.values.table_ids}}"
            task:
              task_key: Ingest_single_table
              spark_python_task:
                python_file: ../src/WorkoutTime_Analytics_Databricks/test.py
                parameters:
                  - --table_id
                  - "{{input}}"

In [12]:
configs_dicts = [{'table_id': 2, 'source_path': '/Volumes/dev/bronze_workout/rw/dataset/gym_logins/', 'source_format': 'csv', 'header': True, 'target_catalog': 'dev', 'target_schema': 'bronze_workout', 'table_version': 1, 'target_table': 'gym_logins', 'write_mode': 'append', 'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/', 'merge_key': [], 'partition_by': [], 'enabled': True, 'trigger_mode': 'availableNow', 'processing_time': '5 minutes', 'schema_evolution_mode': 'BACKWARD'}, {'table_id': 1, 'source_path': '/Volumes/dev/bronze_workout/rw/dataset/bpm/', 'source_format': 'json', 'header': True, 'target_catalog': 'dev', 'target_schema': 'bronze_workout', 'table_version': 1, 'target_table': 'bpm', 'write_mode': 'append', 'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/', 'merge_key': [], 'partition_by': [], 'enabled': True, 'trigger_mode': 'availableNow', 'processing_time': '5 minutes', 'schema_evolution_mode': 'BACKWARD'}]

configs_dicts

[{'table_id': 2,
  'source_path': '/Volumes/dev/bronze_workout/rw/dataset/gym_logins/',
  'source_format': 'csv',
  'header': True,
  'target_catalog': 'dev',
  'target_schema': 'bronze_workout',
  'table_version': 1,
  'target_table': 'gym_logins',
  'write_mode': 'append',
  'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/',
  'merge_key': [],
  'partition_by': [],
  'enabled': True,
  'trigger_mode': 'availableNow',
  'processing_time': '5 minutes',
  'schema_evolution_mode': 'BACKWARD'},
 {'table_id': 1,
  'source_path': '/Volumes/dev/bronze_workout/rw/dataset/bpm/',
  'source_format': 'json',
  'header': True,
  'target_catalog': 'dev',
  'target_schema': 'bronze_workout',
  'table_version': 1,
  'target_table': 'bpm',
  'write_mode': 'append',
  'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/',
  'merge_key': [],
  'partition_by': [],
  'enabled': True,
  'trigger_mode': 'availableNow',
  'processing_time': '5 minutes',
  'schema_evolution_mode': 'BACKWAR

In [13]:
table_id = 1
configs_dict = [d for d in configs_dicts if d["table_id"] == table_id][0]
configs_dict

{'table_id': 1,
 'source_path': '/Volumes/dev/bronze_workout/rw/dataset/bpm/',
 'source_format': 'json',
 'header': True,
 'target_catalog': 'dev',
 'target_schema': 'bronze_workout',
 'table_version': 1,
 'target_table': 'bpm',
 'write_mode': 'append',
 'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/',
 'merge_key': [],
 'partition_by': [],
 'enabled': True,
 'trigger_mode': 'availableNow',
 'processing_time': '5 minutes',
 'schema_evolution_mode': 'BACKWARD'}

In [0]:

spark.sql(f"create table dev.bronze_workout.test")

In [0]:

df = spark.read.schema(schema).json(configs_dict["source_path"])
df.show(truncate=False)

In [0]:
df.printSchema()

In [0]:
configs_dict

In [0]:
from pyspark.sql import SparkSession, DataFrame

def load_date(spark, configs_dict,schema) -> DataFrame:
    """Load data from a given path with the provided schema."""

    return (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", configs_dict["source_format"])
            .option("cloudFiles.schemaLocation", configs_dict["checkpoint_path"]+"schema/"+configs_dict["target_table"])
            .option("cloudFiles.schemaEvolutionMode","rescue")
            .option("cloudFiles.maxFilesPerTrigger", 1)
            .schema(schema)
            .load(configs_dict["source_path"])

    )


In [0]:
raw_data = load_date(spark=spark,configs_dict=configs_dict,schema=schema)


In [10]:
def view_streaming_df(df):
    """View streaming DataFrame"""
    return display(
        df,
        checkpointLocation=f"{configs_dict['checkpoint_path']}tmp/{configs_dict['target_table']}"
    )


In [0]:
view_streaming_df(df=raw_data)

In [0]:
def ensure_target_table_exists(spark,configs_dict):
    try:
        print(f"creating table")
        spark.sql(f"DESCRIBE TABLE {configs_dict['target_catalog']}.{configs_dict['target_schema']}.{configs_dict['target_table']}")
        return
    except Exception:
        pass

    spark.sql(f"CREATE TABLE {configs_dict['target_catalog']}.{configs_dict['target_schema']}.{configs_dict['target_table']}")







In [0]:
ensure_target_table_exists(spark,configs_dict)

In [0]:
configs_dict

In [0]:
def merge_to_delta(spark: SparkSession,df: DataFrame,configs_dict):

    ensure_target_table_exists(spark,configs_dict)    
    print(f"merge keys : {configs_dict['merge_key']}")

    target_full_name = (f"{configs_dict['target_catalog']}."
                        f"{configs_dict['target_schema']}."
                        f"{configs_dict['target_table']}")
    
    if not configs_dict["merge_key"]:
        print("merge keys not present")
        return (df.write
            .format("delta") 
            .option("mergeSchema", "true") 
            .mode(configs_dict["write_mode"]) 
            .saveAsTable(target_full_name)
            )

    else:
        df.createOrReplaceTempView("source_table") 

        condition = " AND ".join([f"t.{key} = s.{key}" for key in configs_dict['merge_key']])

        query = f"""
        MERGE INTO {target_full_name} AS t
        USING source_table AS s
        ON {condition}
        WHEN MATCHED THEN
        UPDATE SET * 
        WHEN NOT MATCHED THEN
        INSERT * 
            
        """

        spark.sql(query)


In [0]:
def start_stream(df, configs_dict):
    writer = (
        df.writeStream
          .foreachBatch(lambda batch_df, batch_id:
                        merge_to_delta(spark, batch_df, configs_dict))
         .option("checkpointLocation", configs_dict["checkpoint_path"]+configs_dict["target_table"]) 
          .queryName(configs_dict["target_table"])
    )

    if configs_dict["trigger_mode"] == "availableNow":
        return writer.trigger(availableNow=True).start()
    else:
        return writer.trigger(processingTime=configs_dict["processing_time"]).start()


In [0]:
start_stream(df=raw_data, configs_dict=configs_dict)

In [0]:
rows = spark.table("dev.control.ingest_table_registry").filter("enabled = true").collect()
rows


In [0]:
tables = [row.asDict() for row in rows]
tables

In [0]:
spark.sql("DROP TABLE dev.control.ingest_run_history")

In [0]:
history_table = "ingest_run_history"
query = f"""
CREATE TABLE IF NOT EXISTS {configs_dict["target_catalog"]}.{ingestion_db}.{history_table} (
    
  id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  run_id STRING NOT NULL,
  table_id INT NOT NULL,
  start_ts TIMESTAMP,    
  end_ts TIMESTAMP DEFAULT NULL,      
  duration_seconds BIGINT DEFAULT 0,
  status STRING DEFAULT 'running',     
  error STRING DEFAULT NULL,
  created_by STRING DEFAULT 'gaurav.thagunna',
  created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP()
  )
  TBLPROPERTIES (
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.feature.allowColumnDefaults' = 'supported'
  );    

"""

spark.sql(query)

In [0]:
spark.sql(f"select * from {configs_dict['target_catalog']}.{ingestion_db}.{history_table} ").show()

In [0]:
spark.sql(f"truncate table {configs_dict['target_catalog']}.{ingestion_db}.{history_table}")

In [0]:
import json, time, datetime, uuid

def get_runid():
    return str(uuid.uuid4())

def now_ts():
    return datetime.datetime.utcnow()


def start_run(table_id: str) -> str:
    start_ts = now_ts()
    start_ts_str = start_ts.isoformat()
    run_id = get_runid()



    spark.sql(f"""INSERT INTO {configs_dict['target_catalog']}.{ingestion_db}.{history_table} (run_id, table_id, start_ts)
                  VALUES ('{run_id}','{table_id}', '{start_ts_str}')
                  
                  """)
    
    return start_ts,run_id


In [0]:
start_ts,run_id = start_run(table_id=configs_dict["table_id"])
run_id


In [0]:
spark.sql(f"select * from {configs_dict['target_catalog']}.{ingestion_db}.{history_table} ").show(truncate=False)

In [0]:

end_ts = now_ts()
print(end_ts)

duration_seconds = int((end_ts - start_ts).total_seconds())
print(duration_seconds)

end_ts_str = end_ts.isoformat()






In [0]:
spark.sql("select * from dev.control.ingest_table_registry").show(truncate=False)

In [0]:
def finish_run_success(run_id, end_ts, table_id, duration_seconds):
    spark.sql(f"""
              UPDATE {configs_dict['target_catalog']}.{ingestion_db}.{history_table} 
              SET end_ts = TIMESTAMP'{end_ts}', status='success', 
              duration_seconds={duration_seconds}
              WHERE run_id='{run_id}'""")
    
    spark.sql(f"""
              MERGE INTO dev.control.ingest_table_registry tr USING (SELECT '{table_id}' as table_id) s ON tr.table_id = s.table_id WHEN MATCHED THEN UPDATE SET 
              last_success = TIMESTAMP'{end_ts}', 
              last_error = NULL, 
              consecutive_failures = 0, 
              last_run_duration_seconds = '{duration_seconds}'""")

In [0]:
finish_run_success(run_id=run_id, end_ts=end_ts, table_id=configs_dict["table_id"], duration_seconds=duration_seconds)

In [0]:
spark.table(f"{configs_dict['target_catalog']}.{ingestion_db}.{history_table} ").show(truncate=False)

In [0]:
spark.table(f"dev.control.ingest_table_registry ").show(truncate=False)

In [0]:
spark.sql("select * from dev.control.ingest_run_history ").show()

In [0]:



def finish_run_failure(run_id, table_id, err_msg):
    end_ts = now_ts()

    duration_seconds = int((end_ts - start_ts).total_seconds())


    end_ts_str = end_ts.isoformat()
    esc = err_msg.replace("'", "''")[:2000]
    spark.sql(f"""UPDATE dev.control.ingest_run_history 
              SET end_ts = TIMESTAMP'{end_ts_str}', status='failed', error = '{err_msg}', duration_seconds = '{duration_seconds}' WHERE run_id = '{run_id}'""")
    
    spark.sql(f"""MERGE INTO dev.control.ingest_table_registry tr USING (SELECT '{table_id}' as table_id) s 
              ON tr.table_id = s.table_id WHEN MATCHED THEN 
              UPDATE SET last_error = '{err_msg}', 
              consecutive_failures = coalesce(tr.consecutive_failures,0)+1, 
              last_run_duration_seconds = '{duration_seconds}'""")


In [0]:
finish_run_failure(run_id=run_id, table_id=configs_dict["table_id"], err_msg="Error")

In [0]:
spark.sql("select * from dev.control.ingest_run_history ").show(truncate=False)

In [0]:
spark.sql("select * from dev.control.ingest_table_registry ").show(truncate=False)

In [0]:
CREATE TABLE control.bronze_tables (
  table_name STRING,
  source_path STRING,
  format STRING,
  header BOOLEAN,
  schema_id STRING,
  merge_key STRING,
  partition_by STRING,
  enabled BOOLEAN,
  trigger_mode STRING,       
  processing_time STRING,  
  owner STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA;

In [0]:
spark.sql("select current_version()").show(truncate=False)


In [0]:
import sys
print(sys.version)


In [0]:
!pwd

In [0]:
print("hello")

In [1]:
spark.sql("select * from dev.control.schema_registry").show()

+---+--------+----------+-------+---------+-------------+--------------------+---------------+--------------------+
| id|table_id|table_name|version|is_latest|schema_format|   schema_definition|     created_by|          created_at|
+---+--------+----------+-------+---------+-------------+--------------------+---------------+--------------------+
|  2|       1|       bpm|      2|     true|         json|{"fields":[{"meta...|gaurav.thagunna|2026-01-30 16:44:...|
|  1|       1|       bpm|      1|     true|         json|{"fields":[{"meta...|gaurav.thagunna|2026-01-27 12:54:...|
+---+--------+----------+-------+---------+-------------+--------------------+---------------+--------------------+



In [6]:
spark.sql("select count(1) from dev.bronze_workout.bpm").show(truncate=False)

+--------+
|count(1)|
+--------+
|507602  |
+--------+



In [8]:
spark.sql("select count(1) from dev.bronze_workout.bpm").show(truncate=False)

+--------+
|count(1)|
+--------+
|507602  |
+--------+



In [5]:
spark.sql("select * from dev.bronze_workout.bpm").show(truncate=False)

+------+------+---------+----------+-----+----------------------------------------+-------------+
|key   |offset|partition|timestamp |topic|value                                   |_rescued_data|
+------+------+---------+----------+-----+----------------------------------------+-------------+
|175406|1900  |0        |1678581900|bpm  |{175406, 93.17670828729, 1678581900}    |NULL         |
|175406|1901  |1        |1678581901|bpm  |{175406, 70.57985125966808, 1678581901} |NULL         |
|118440|1901  |1        |1678581901|bpm  |{118440, 38.38741068141134, 1678581901} |NULL         |
|175406|1902  |2        |1678581902|bpm  |{175406, 48.762405497753804, 1678581902}|NULL         |
|118440|1902  |2        |1678581902|bpm  |{118440, 76.15689560826479, 1678581902} |NULL         |
|175406|1903  |3        |1678581903|bpm  |{175406, 75.23276626007171, 1678581903} |NULL         |
|118440|1903  |3        |1678581903|bpm  |{118440, 46.201859410330734, 1678581903}|NULL         |
|118440|1904  |4    

In [ ]:
%sql
/Volumes/dev/bronze_workout/rw

select * from json.'/Volumes/dev/bronze_workout/rw/bpm/'

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near ''/Volumes/dev/bronze_workout/rw/bpm/''. SQLSTATE: 42601 (line 1, pos 19)

== SQL ==
select * from json.'/Volumes/dev/bronze_workout/rw/bpm/'
-------------------^^^


JVM stacktrace:
org.apache.spark.sql.catalyst.parser.ParseException
	at org.apache.spark.sql.catalyst.parser.ParseException.withCommand(parsers.scala:479)
	at org.apache.spark.sql.catalyst.parser.AbstractParser.parse(parsers.scala:120)
	at org.apache.spark.sql.execution.SparkSqlParser.parse(SparkSqlParser.scala:167)
	at org.apache.spark.sql.catalyst.parser.AbstractSqlParser.parsePlan(AbstractSqlParser.scala:118)
	at org.apache.spark.sql.classic.SparkSession.$anonfun$sql$7(SparkSession.scala:842)
	at org.apache.spark.sql.catalyst.QueryPlanningTracker$.withTracker(QueryPlanningTracker.scala:265)
	at org.apache.spark.sql.classic.SparkSession.$anonfun$sql$6(SparkSession.scala:842)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:11

In [1]:
%sql

select * from dev.control.schema_registry;

,id,table_id,table_name,version,is_latest,schema_format,schema_definition,created_by,created_at
0,2,1,bpm,2,True,json,"{""fields"":[{""metadata"":{},""name"":""key"",""nullable"":true,""type"":""long""},{""metadata"":{},""name"":""offset"",""nullable"":true,""type"":""long""},{""metadata"":{},""name"":""partition"",""nullable"":true,""type"":""long""},{""metadata"":{},""name"":""timestamp"",""nullable"":true,""type"":""long""},{""metadata"":{},""name"":""topic"",""nullable"":true,""type"":""string""},{""metadata"":{},""name"":""value"",""nullable"":true,""type"":{""fields"":[{""metadata"":{},""name"":""device_id"",""nullable"":true,""type"":""long""},{""metadata"":{},""name"":""heartrate"",""nullable"":true,""type"":""double""},{""metadata"":{},""name"":""time"",""nullable"":true,""type"":""long""}],""type"":""struct""}}],""type"":""struct""}",gaurav.thagunna,2026-01-30 16:44:54.993144
1,1,1,bpm,1,True,json,"{""fields"":[{""metadata"":{},""name"":""key"",""nullable"":true,""type"":""long""},{""metadata"":{},""name"":""offset"",""nullable"":true,""type"":""long""},{""metadata"":{},""name"":""partition"",""nullable"":true,""type"":""long""},{""metadata"":{},""name"":""timestamp"",""nullable"":true,""type"":""long""},{""metadata"":{},""name"":""topic"",""nullable"":true,""type"":""string""},{""metadata"":{},""name"":""value"",""nullable"":true,""type"":{""fields"":[{""metadata"":{},""name"":""device_id"",""nullable"":true,""type"":""long""},{""metadata"":{},""name"":""heartrate"",""nullable"":true,""type"":""double""},{""metadata"":{},""name"":""time"",""nullable"":true,""type"":""long""}],""type"":""struct""}}],""type"":""struct""}",gaurav.thagunna,2026-01-27 12:54:10.025041


In [2]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

In [24]:
volume_path = "/Volumes/dev/bronze_workout/checkpoints/bpm"
for item in w.files.list_directory_contents(volume_path):
  print(item.path)

/Volumes/dev/bronze_workout/checkpoints/bpm/commits/
/Volumes/dev/bronze_workout/checkpoints/bpm/offsets/
/Volumes/dev/bronze_workout/checkpoints/bpm/sources/


In [19]:
volume_path = "/Volumes/dev/bronze_workout/checkpoints/bpm/commits/"

w.files.delete(volume_path)


BadRequest: Paths ending in the '/' character represent directories. This API does not support operations on directories.

In [8]:
spark.sql("select * from dev.bronze_workout.gym_logins").show(truncate=False)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `dev`.`bronze_workout`.`gym_logins` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [dev, bronze_workout, gym_logins], [], false


JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.tableNotFound(package.scala:94)
	at org.apache.spark.sql.catalyst.analysis.RelationResolution$.throwTableNotFound(RelationResolution.scala:1017)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$2(CheckAnalysis.scala:355)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$2$adapted(Chec

In [4]:
spark.sql("drop TABLE dev.bronze_workout.bpm")

""


In [3]:
spark.sql("drop TABLE dev.bronze_workout.gym_logins")

""


In [5]:
dbutils.fs.rm("/Volumes/dev/bronze_workout/checkpoints/bpm",True)


True

In [2]:
dbutils.fs.rm("/Volumes/dev/bronze_workout/checkpoints/gym_logins",True)


True

In [2]:
dbutils.fs.ls("/Volumes/dev/bronze_workout/checkpoints")


[FileInfo(path='/Volumes/dev/bronze_workout/checkpoints/bpm/', name='', size=None, modificationTime=None),
 FileInfo(path='/Volumes/dev/bronze_workout/checkpoints/gym_logins/', name='', size=None, modificationTime=None),
 FileInfo(path='/Volumes/dev/bronze_workout/checkpoints/schema/', name='', size=None, modificationTime=None),
 FileInfo(path='/Volumes/dev/bronze_workout/checkpoints/tmp/', name='', size=None, modificationTime=None)]

In [1]:
spark.sql("select count(*) from dev.bronze_workout.bpm")

,count(*)
0,507602


In [8]:
spark.sql("select * from dev.bronze_workout.bpm")

,key,offset,partition,timestamp,topic,value,_rescued_data,ingestion_timestamp
0,175406,1900,0,1678581900,bpm,"{'device_id': 175406, 'heartrate': 93.17670828729, 'time': 1678581900}",NaN,2026-02-03 15:14:00.551
1,175406,1901,1,1678581901,bpm,"{'device_id': 175406, 'heartrate': 70.57985125966808, 'time': 1678581901}",NaN,2026-02-03 15:14:00.551
2,118440,1901,1,1678581901,bpm,"{'device_id': 118440, 'heartrate': 38.38741068141134, 'time': 1678581901}",NaN,2026-02-03 15:14:00.551
3,175406,1902,2,1678581902,bpm,"{'device_id': 175406, 'heartrate': 48.762405497753804, 'time': 1678581902}",NaN,2026-02-03 15:14:00.551
4,118440,1902,2,1678581902,bpm,"{'device_id': 118440, 'heartrate': 76.15689560826479, 'time': 1678581902}",NaN,2026-02-03 15:14:00.551
5,175406,1903,3,1678581903,bpm,"{'device_id': 175406, 'heartrate': 75.23276626007171, 'time': 1678581903}",NaN,2026-02-03 15:14:00.551
6,118440,1903,3,1678581903,bpm,"{'device_id': 118440, 'heartrate': 46.201859410330734, 'time': 1678581903}",NaN,2026-02-03 15:14:00.551
7,118440,1904,4,1678581904,bpm,"{'device_id': 118440, 'heartrate': 39.291813058803584, 'time': 1678581904}",NaN,2026-02-03 15:14:00.551
8,175406,1904,4,1678581904,bpm,"{'device_id': 175406, 'heartrate': 85.52990991565872, 'time': 1678581904}",NaN,2026-02-03 15:14:00.551
9,175406,1905,5,1678581905,bpm,"{'device_id': 175406, 'heartrate': 49.42239653650089, 'time': 1678581905}",NaN,2026-02-03 15:14:00.551


In [10]:
spark.sql("Truncate table dev.bronze_workout.bpm")

""


In [3]:
spark.sql("select * from dev.control.schema_registry").show(truncate=False)

+---+--------+----------+-------+---------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+--------------------------+
|id |table_id|table_name|version|is_latest|schema_format|schema_definition                                                                                                                                                                                                               

In [2]:
spark.sql("select * from dev.control.ingest_run_history").show(truncate=False)

+---+------------------------------------+--------+--------------------------+--------------------------+----------------+-------+-----+---------------+--------------------------+
|id |run_id                              |table_id|start_ts                  |end_ts                    |duration_seconds|status |error|created_by     |created_at                |
+---+------------------------------------+--------+--------------------------+--------------------------+----------------+-------+-----+---------------+--------------------------+
|1  |004a397e-d3fc-45f7-9be2-af2ffe85983d|1       |2026-01-30 15:19:17.865189|2026-01-30 15:26:56.735702|458             |failed |Error|gaurav.thagunna|2026-01-30 15:19:18.336157|
|2  |4870f8bd-c029-4bfa-99d0-299c6bf152be|1       |2026-02-03 15:13:43.976652|2026-02-03 15:13:53.557385|9               |success|NULL |gaurav.thagunna|2026-02-03 15:13:44.717588|
|3  |67d5981a-bd08-4713-a5b3-aed165c175bf|1       |2026-02-03 15:24:18.715594|2026-02-03 15:24:23.43

In [1]:
spark.sql("select * from dev.control.ingest_table_registry").show(truncate=False)

NameError: name 'spark' is not defined

In [ ]:
query = f"""

INSERT INTO dev.control.schema_registry (
  table_id,
  table_name,
  version,
  is_latest,
  schema_format,
  schema_definition,
  created_by
)
VALUES (
  1,
  'bpm',
  3,
  TRUE,
  'json',
  '{schema.json()}',
  'gaurav.thagunna'
)

"""

spark.sql(query)

# A sample job for WorkoutTime_Analytics_Databricks.

resources:
  jobs:
    Workout_Analytics:
      name: Workout_Analytics

      trigger:
        # Run this job every day, exactly one day from the last run; see https://docs.databricks.com/api/workspace/jobs/create#trigger
        periodic:
          interval: 1
          unit: DAYS

      parameters:
        - name: catalog
          default: ${var.catalog}
        - name: schema
          default: ${var.schema}
        - name: ingestion_db
          default: control
        - name: ingestion_table
          default: ingest_table_registry
        - name: schema_registry_table
          default: schema_registry
        - name: bronze_db
          default: bronze_workout
        - name: history_table
          default: ingest_run_history

      tasks:
        - task_key: Ingestion_master
          spark_python_task:
            python_file: ../src/WorkoutTime_Analytics_Databricks/setup_ingestion_master.py

        - task_key: notebook_task
          notebook_task:
            notebook_path: ../src/sample_notebook.ipynb
        - task_key: python_wheel_task
          depends_on:
            - task_key: notebook_task
          python_wheel_task:
            package_name: WorkoutTime_Analytics_Databricks
            entry_point: main
            parameters:
              - "--catalog"
              - "${var.catalog}"
              - "--schema"
              - "${var.schema}"
          environment_key: default
        - task_key: refresh_pipeline
          depends_on:
            - task_key: notebook_task
          pipeline_task:
            pipeline_id: ${resources.pipelines.WorkoutTime_Analytics_Databricks_etl.id}

      environments:
        - environment_key: default
          spec:
            environment_version: "4"
            dependencies:
              # By default we just include the .whl file generated for the WorkoutTime_Analytics_Databricks package.
              # See https://docs.databricks.com/dev-tools/bundles/library-dependencies.html
              # for more information on how to add other libraries.
              - ../dist/*.whl





In [ ]:
spark